# Local full run: Qwen3-4B LoRA + riskpo

Giữ thuật toán, model, dữ liệu, LoRA, thinking, batch, rollout, learning rate,
offload và topology **4 GPU** của notebook nguồn `RiskPO_Original_Qwen3_4B_4xA100.ipynb`.
`full` vẫn là LoRA, không chuyển sang cập nhật toàn bộ trọng số.

Thay đổi vận hành theo yêu cầu chạy local:
- Chạy liên tục hết **200 bước GSM8K/easymath hoặc 500 bước DAPO**.
- Không mount/copy Drive; không giới hạn phiên 40 bước và không resume run Colab cũ.
- Không lưu mỗi 5 bước. Lưu checkpoint local ở cuối run; không tự xóa các run cũ.
- Đánh giá split test gốc **một lần ở cuối**, lưu metrics và từng đáp án.
- Sau training, gộp LoRA vào đúng base snapshot trên CPU, lưu model độc lập trong `final_model/`.

Máy đích Linux/CUDA, ổ local bền vững và GPU tương thích BF16/FlashAttention-2.
Chạy lần lượt các cell; không chạy notebook này trên Windows hiện tại.
Mỗi run dùng thư mục mới. Phần chạy full cuối notebook chỉ bắt đầu khi chạy cell training.
Nếu tiến trình dừng trước lần lưu cuối thì chưa có checkpoint mới để tiếp tục.

## 1. Đường dẫn local và tham số gốc

Đặt `LOCAL_ROOT` thành ổ local đủ chỗ, ví dụ `/data/riskpo-local`; mặc định là
`riskpo_local_runtime` dưới thư mục notebook. Tất cả data/cache/log/checkpoint/model
được giữ ở máy đích. Không dùng `.env` hoặc API Viettel.

Gộp model trên CPU cần RAM cho base BF16, adapter và bộ nhớ tạm; dung lượng disk
phải đủ base snapshot + checkpoint + merged model. Notebook in RAM/disk trước khi chạy.

In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import datetime
import hashlib
import warnings

REPO_URL = "https://github.com/Belldenchoi/RiskPO.git"
REPO_REF = "68ed7c5898b009fe7e392c2b01feb85b3bb2ff18"
DATASET = "gsm8k"  # Chỉ GSM8K train/test; tùy chọn cũ: "easymath", "dapo"
METHOD = 'riskpo'
LORA_RANK = 8
LORA_ALPHA = 16
TRAIN_BATCH_SIZE = 20
PPO_MINI_BATCH_SIZE = 20
USE_REFERENCE_MODEL = False
MODEL_ID = "Qwen/Qwen3-4B"
ENABLE_THINKING = False  # Qwen3: False/True; model khác: None (template mặc định)
RUN_PROFILE = "full"





LOCAL_ROOT = Path(os.environ.get("RISKPO_LOCAL_ROOT", str(Path.cwd() / "riskpo_local_runtime"))).expanduser().resolve()
WORK_ROOT = LOCAL_ROOT
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "RiskPO-local-riskpo-4gpu"
PY = WORK_ROOT / "riskpo-env/bin/python"
DATA_DIR = WORK_ROOT / "data/riskpo_reference"
ENV = dict(os.environ, VLLM_USE_V1="0", HYDRA_FULL_ERROR="1",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
assert DATASET in {"gsm8k", "easymath", "dapo"}
assert METHOD == "riskpo", "Notebook này dành cho baseline RiskPO gốc"
assert RUN_PROFILE == "full"
REQUIRED_GPUS = 4
if USE_REFERENCE_MODEL:
    MODEL_ID = ("Qwen/Qwen2.5-1.5B-Instruct" if DATASET != "dapo"
                else "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
assert ENABLE_THINKING is None or type(ENABLE_THINKING) is bool
if ENABLE_THINKING is not None and not MODEL_ID.startswith("Qwen/Qwen3-"):
    raise ValueError("Thinking override này chỉ dành cho Qwen3; model khác đặt ENABLE_THINKING=None.")

def run(args, **kwargs):
    kwargs.setdefault("env", ENV)
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

def run_py(source):
    return run([PY, "-c", source], cwd=REPO)

def check_gpu_count():
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True
    )
    print(gpu_info)
    if len(gpu_info.strip().splitlines()) < REQUIRED_GPUS:
        raise RuntimeError(
            "Cần bốn GPU A100 trên cùng máy Linux. Kiểm tra GPU allocation trước khi cài đặt."
        )
    return gpu_info

# Dừng sớm nếu không có GPU; bước 4 kiểm tra thêm khả năng BF16/FlashAttention.
if not sys.platform.startswith("linux"):
    raise RuntimeError("Notebook training này yêu cầu Linux/CUDA.")
ENV.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,3")
GPU_INFO = check_gpu_count()
run(["free", "-h"])
run(["df", "-h", WORK_ROOT])
print("Model:", MODEL_ID, "| Method:", METHOD, "| Dataset:", DATASET,
      "| enable_thinking:", ENABLE_THINKING)
ENV["HF_HOME"] = str(WORK_ROOT / "hf_cache")

## 2. Clone riêng, Python 3.10 và dependencies như notebook nguồn

In [ ]:
if not REPO.exists():
    run(["git", "clone", REPO_URL, REPO])
    if REPO_REF != "main":
        run(["git", "checkout", "--detach", REPO_REF], cwd=REPO)
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"{REPO} đã tồn tại nhưng không phải Git repo.")
print("Remote:")
run(["git", "remote", "-v"], cwd=REPO)
run(["git", "log", "-1", "--oneline"], cwd=REPO)
core = (REPO / "verl/trainer/ppo/core_algos.py").read_text()
assert "compute_grpo_bundle_rvar_outcome_advantage_quantile_tracking" in core, "Thiếu RiskPO estimator."
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual_commit != REPO_REF:
    raise RuntimeError("Clone hiện có không khớp REPO_REF; dùng WORK_ROOT mới hoặc kiểm tra code trước khi tiếp tục.")

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv")
assert UV, "Không tìm thấy uv sau cài đặt."
run([UV, "python", "install", "3.10"])
if not PY.exists():
    run([UV, "venv", "--python", "3.10", "--seed", PY.parent.parent])
run([PY, "--version"])

In [ ]:
run([UV, "pip", "install", "--python", PY,
     "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
     "--index-url", "https://download.pytorch.org/whl/cu124"])

pins = [
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "vllm==0.8.5.post1", "transformers==4.51.3",
    "peft==0.15.2", "accelerate==1.6.0", "ray[default]==2.43.0",
    "tensordict==0.8.3", "torchdata==0.11.0",
    "datasets==3.6.0", "numpy==1.26.4", "hydra-core==1.3.2",
]
wheel = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/"
    "v2.7.4.post1/"
    "flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-cp310-cp310-linux_x86_64.whl"
)
run([UV, "pip", "install", "--python", PY, *pins,
     wheel, "tensorboard", "-e", str(REPO) + "[math]"])
run([UV, "pip", "check", "--python", PY])

In [ ]:
def run_py(source):
    result = subprocess.run(
        [str(PY), "-u", "-c", source],
        cwd=str(REPO),
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Kiểm tra thất bại, exit={result.returncode}. "
            "Xem traceback ngay phía trên."
        )
    return result

## 3. Kiểm tra GPU, thư viện và giữ nguyên scorer final_numeric_v2

In [ ]:
GSM8K_REWARD_SOURCE = r'''# Copyright 2024 Bytedance Ltd. and/or its affiliates
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import re
from decimal import Decimal, InvalidOperation

_SOLUTION_CLIP_CHARS = 300
GSM8K_SCORER_VERSION = "final_numeric_v2"
_ANSWER_MARKER = re.compile(r"####(?!#)|\\boxed\b")
_NUMBER = re.compile(r"[+-]?(?:(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?")


def _numeric_token(value):
    """Accept a scalar number, not units, expressions, or arbitrary answer text."""
    token = str(value).strip()
    if token.startswith("$") and token.endswith("$"):
        token = token[1:-1].strip()
    if len(token) > 256 or _NUMBER.fullmatch(token) is None:
        return None
    return token.replace(",", "")


def _extract_strict(solution_str):
    # Only the final-response portion counts for models with explicit thinking tags.
    # A correct intermediate answer in <think> must not earn reward when the final
    # response is missing, truncated, or wrong.
    final_response = solution_str.rsplit("</think>", 1)[-1]
    if "<think>" in final_response:
        return None

    # Search the whole final response: the answer may be followed by >300 chars.
    # Do not fall back to an earlier correct answer if the last marker is invalid.
    marker = None
    for match in _ANSWER_MARKER.finditer(final_response):
        marker = match
    if marker is None:
        return None
    tail = final_response[marker.end():].lstrip(" \t")
    if marker.group().startswith("####"):
        return _numeric_token(tail.splitlines()[0] if tail else "")

    tail = tail.lstrip()
    if not tail.startswith("{"):
        return None
    depth = 0
    for index, char in enumerate(tail):
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return _numeric_token(tail[1:index])
    return None


def extract_solution(solution_str, method="strict"):
    assert method in ["strict", "flexible"]

    if method == "strict":
        return _extract_strict(solution_str)

    # Preserve the legacy flexible mode; training uses strict explicit markers.
    if len(solution_str) > _SOLUTION_CLIP_CHARS:
        solution_str = solution_str[-_SOLUTION_CLIP_CHARS:]

    answer = re.findall("(\\-?[0-9\\.\\,]+)", solution_str)
    final_answer = None
    for candidate in reversed(answer):
        if candidate not in ["", "."]:
            final_answer = candidate
            break
    return final_answer


def compute_score(solution_str, ground_truth, method="strict", format_score=0.0, score=1.0):
    """The scoring function for GSM8k.

    Reference: Trung, Luong, et al. "Reft: Reasoning with reinforced fine-tuning." Proceedings of the 62nd Annual
    Meeting of the Association for Computational Linguistics (Volume 1: Long Papers). 2024.

    Args:
        solution_str: the solution text
        ground_truth: the ground truth
        method: 'strict' accepts a final boxed scalar or #### numeric answer;
            'flexible' retains the legacy last-number heuristic.
        format_score: the score for the format
        score: the score for the correct answer
    """
    answer = extract_solution(solution_str=solution_str, method=method)
    answer = _numeric_token(answer) if answer is not None else None
    expected = _numeric_token(ground_truth)
    if answer is None or expected is None:
        return 0
    try:
        # Exact decimal equality, not an approximate float tolerance or eval().
        # This treats 10, 10.0 and 1e1 as the same numerical answer.
        correct = Decimal(answer) == Decimal(expected)
    except InvalidOperation:
        return 0
    return score if correct else format_score

def compute_dataset_score(data_source, solution_str, ground_truth, extra_info=None, **kwargs):
    """Use the corrected GSM8K scorer; leave MATH/DAPO and other routes unchanged."""
    if data_source == "openai/gsm8k":
        return compute_score(solution_str, ground_truth)
    from verl.utils.reward_score import default_compute_score
    return default_compute_score(
        data_source=data_source, solution_str=solution_str,
        ground_truth=ground_truth, extra_info=extra_info, **kwargs
    )
'''
GSM8K_REWARD_PATH = WORK_ROOT / "riskpo_reward_final_numeric_v2.py"
if GSM8K_REWARD_PATH.exists():
    if GSM8K_REWARD_PATH.read_text(encoding="utf-8") != GSM8K_REWARD_SOURCE:
        raise RuntimeError("Custom reward path đã có nội dung khác; đổi tên file để giữ bản cũ.")
else:
    GSM8K_REWARD_PATH.write_text(GSM8K_REWARD_SOURCE, encoding="utf-8")

run_py("REQUIRED_GPUS = " + repr(REQUIRED_GPUS) + "\n"
       + "REWARD_PATH = " + repr(str(GSM8K_REWARD_PATH)) + "\n" + r'''
import torch
import transformers
import vllm
import flash_attn
import tensorboard
from verl.trainer.ppo.core_algos import (
    compute_grpo_bundle_rvar_outcome_advantage_quantile_tracking, compute_policy_loss,
)
import importlib.util
spec = importlib.util.spec_from_file_location("riskpo_reward_v2_check", REWARD_PATH)
reward = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reward)
default_compute_score = reward.compute_dataset_score

print("torch", torch.__version__, "CUDA", torch.version.cuda)
print("transformers", transformers.__version__, "vllm", vllm.__version__)
print("flash_attn", flash_attn.__version__)
assert torch.cuda.device_count() == REQUIRED_GPUS == 4, "Expose đúng bốn GPU qua CUDA_VISIBLE_DEVICES."
for i in range(REQUIRED_GPUS):
    print(i, torch.cuda.get_device_name(i))
    assert "A100" in torch.cuda.get_device_name(i), "Cấu hình này dành cho bốn A100."
    assert torch.cuda.get_device_capability(i)[0] >= 8, "Cần GPU hỗ trợ FlashAttention-2/BF16."
assert default_compute_score("openai/gsm8k", "#### 42", "42") == 1
assert default_compute_score("openai/gsm8k", r"\boxed{42}", "42") == 1
assert default_compute_score("openai/gsm8k", r"\boxed{41}", "42") == 0
assert default_compute_score("openai/gsm8k", r"<think>\boxed{42}</think>No final answer.", "42") == 0
assert default_compute_score("openai/gsm8k", r"<think>\boxed{41}</think>Final: \boxed{42}", "42") == 1
print("GSM8K scorer:", reward.GSM8K_SCORER_VERSION)
assert default_compute_score(
    "DigitalLearningGmbH/MATH-lighteval", r"\boxed{42}", "42"
) == 1
print("Import/GPU/reward checks passed; chưa kiểm tra đủ VRAM cho training.")
''')

In [ ]:
DISTRIBUTED_PREFLIGHT_SOURCE = 'import os\nfrom datetime import timedelta\nimport torch\nimport torch.distributed as dist\nrank = int(os.environ["LOCAL_RANK"])\ntorch.cuda.set_device(rank)\ndist.init_process_group("nccl", timeout=timedelta(seconds=120))\ntry:\n    assert dist.get_world_size() == 4\n    value = torch.tensor([float(dist.get_rank() + 1)], device=f"cuda:{rank}")\n    dist.all_reduce(value)\n    torch.cuda.synchronize()\n    assert value.item() == 10.0\n    print(f"NCCL_OK rank={dist.get_rank()} GPU={torch.cuda.get_device_name(rank)} sum={value.item()}", flush=True)\nfinally:\n    dist.destroy_process_group()\n'
preflight_path = WORK_ROOT / "riskpo_four_gpu_preflight.py"
preflight_path.write_text(DISTRIBUTED_PREFLIGHT_SOURCE, encoding="utf-8")
run([PY, "-m", "torch.distributed.run", "--standalone", "--nnodes=1",
     "--nproc-per-node=4", preflight_path], cwd=REPO, timeout=180)

## 4. Chuẩn bị dữ liệu và cố định base snapshot dùng cho cả train/export

In [ ]:
OUTPUT_ROOT = WORK_ROOT / "runs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print("Local outputs:", OUTPUT_ROOT)
print("Local checkpoints:", CHECKPOINT_ROOT)

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = DATA_DIR / "raw"
GSM8K_PREPROCESS_SOURCE = r'''
from pathlib import Path
import re
import datasets

def make_gsm8k_row(example, idx, split):
    question_raw, answer_raw = example['question'], example['answer']
    match = re.search(r'#### (\-?[0-9\.\,]+)', answer_raw)
    assert match is not None, 'GSM8K answer missing #### numeric ground truth'
    solution = match.group(1).replace(',', '')
    instruction = r"Let's think step by step and output the final answer within \boxed{}."
    return {
        'data_source': 'openai/gsm8k',
        'prompt': [{'role': 'user', 'content': question_raw + ' ' + instruction}],
        'ability': 'math',
        'reward_model': {'style': 'rule', 'ground_truth': solution},
        'extra_info': {'split': split, 'index': idx, 'answer': answer_raw, 'question': question_raw},
    }

def prepare_gsm8k(output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    missing = [split for split in ('train', 'test') if not (output_dir / (split + '.parquet')).exists()]
    if not missing:
        print('Reuse existing GSM8K train/test; contents checked in step 8.')
        return
    dataset = datasets.load_dataset('openai/gsm8k', 'main')
    for split in missing:
        processed = dataset[split].map(make_gsm8k_row, with_indices=True, fn_kwargs={'split': split})
        processed.to_parquet(str(output_dir / (split + '.parquet')))
        print('GSM8K', split, 'rows:', len(processed))
'''
if DATASET == "gsm8k":
    run_py(GSM8K_PREPROCESS_SOURCE + f"\nprepare_gsm8k({str(RAW_DIR / 'gsm8k')!r})\n")
elif DATASET == "easymath":
    for name in ("math", "gsm8k"):
        (RAW_DIR / name).mkdir(parents=True, exist_ok=True)
    run([PY, "data_processing/download_easymath.py",
         "--math_local_dir", RAW_DIR / "math",
         "--gsm8k_local_dir", RAW_DIR / "gsm8k"], cwd=REPO)
else:
    run([PY, "data_processing/download_dapomath.py",
         "--output_dir", RAW_DIR], cwd=REPO)

In [ ]:
snapshot_file = WORK_ROOT / (MODEL_ID.replace("/", "--") + "_snapshot.json")
run_py("MODEL_ID = " + repr(MODEL_ID) + "\nSNAPSHOT_FILE = " + repr(str(snapshot_file)) + "\n" + r'''
import json
from pathlib import Path
from huggingface_hub import snapshot_download
snapshot = snapshot_download(repo_id=MODEL_ID)
Path(SNAPSHOT_FILE).write_text(json.dumps({"model_id": MODEL_ID, "path": snapshot,
                                        "revision": Path(snapshot).name}, indent=2), encoding="utf-8")
''')
BASE_SNAPSHOT = json.loads(snapshot_file.read_text(encoding="utf-8"))
print("Exact base snapshot:", BASE_SNAPSHOT)

## 5. Cấu hình full local

Các builder training được giữ nguyên từ notebook nguồn. Chỉ áp dụng thay đổi
lưu trữ, chạy liên tục và đánh giá cuối run sau khi gọi builder. Không có helper
durable, copy backup, prune hoặc session hook. Checkpoint cuối lưu cả model,
optimizer và extra của từng rank, cùng dataloader; không phải chỉ adapter.

In [ ]:
def build_reference_config(dataset, method, model_id, data_root, checkpoint_dir, run_name):
    """Match executable overrides in the RiskPO scripts, then apply the chosen method."""
    if dataset not in {"gsm8k", "easymath", "dapo"} or method != "riskpo":
        raise ValueError("Unknown dataset or method")
    hard = dataset == "dapo"
    raw = Path(data_root) / "raw"
    train_files = ([str(raw / "dapo_aime2024/dapo-math-17k.parquet")] if hard else
                   [str(raw / "gsm8k/train.parquet"), str(raw / "math/train.parquet")])
    val_files = ([str(raw / "dapo_aime2024/aime-2024.parquet")] if hard else
                 [str(raw / "gsm8k/test.parquet"), str(raw / "math/test.parquet")])
    if dataset == "gsm8k":
        train_files = [str(raw / "gsm8k/train.parquet")]
        val_files = [str(raw / "gsm8k/test.parquet")]
    algorithm = {
        "adv_estimator": "grpo_bundle_RVaR_quantile_tracking",
        "quantile_down": 0.2, "quantile_up": 0.8 if hard else 0.9,
        "bundle_size": 5, "lr_q": 0.1,
        "credit_assign_mode": "sum-mean" if hard else "std",
        "use_q_track_mode": "track", "norm_adv_by_std_in_grpo": True,
        "use_mixing_risk_measure": True, "w_mix": 1.5,
        "use_mean_as_baseline": False, "natural_baseline_base": True,
        "natural_baseline_adv": not hard,
        "quantile_tracking": True, "use_kl_in_reward": False,
    }
    actor = {
        "optim": {"lr": 1e-6},
        "ppo_mini_batch_size": 128 if hard else 512,
        "ppo_micro_batch_size_per_gpu": 64,
        "use_kl_loss": False, "kl_loss_coef": 0.001,
        "kl_loss_type": "low_var_kl", "entropy_coeff": 0,
        "fsdp_config": {"param_offload": hard, "optimizer_offload": False},
        "policy_loss": {"loss_mode": "vanilla"},
        "loss_agg_mode": "token-mean",
    }
    rollout = {
        "name": "vllm", "mode": "sync",
        "log_prob_micro_batch_size_per_gpu": 16 if hard else 64,
        "tensor_model_parallel_size": 2, "gpu_memory_utilization": 0.8,
        "n": 10 if hard else 5,
        "temperature": 1.0, "top_p": 1.0, "top_k": -1,
        "val_kwargs": {"do_sample": False, "n": 1,
                       "temperature": 0.0, "top_p": 1.0, "top_k": -1},
    }
    ref = {
        "log_prob_micro_batch_size_per_gpu": 16 if hard else 64,
        "fsdp_config": {"param_offload": True},
    }
    if hard:
        actor.update(use_dynamic_bsz=False, ppo_max_token_len_per_gpu=4096,
                     ulysses_sequence_parallel_size=4)
        ref.update(log_prob_use_dynamic_bsz=False, log_prob_max_token_len_per_gpu=4096,
                   ulysses_sequence_parallel_size=4)
        rollout.update(log_prob_use_dynamic_bsz=False,
                       log_prob_max_token_len_per_gpu=4096, max_num_batched_tokens=4096)
    return {
        "defaults": ["ppo_trainer", "_self_"],
        "algorithm": algorithm,
        "data": {
            "train_files": train_files, "val_files": val_files,
            "train_batch_size": 512 if hard else 1024,
            "max_prompt_length": 1024, "max_response_length": 3072 if hard else 1024,
            "filter_overlong_prompts": True, "truncation": "error",
        },
        "actor_rollout_ref": {
            "model": {"path": model_id, "lora_rank": 0,
                      "use_remove_padding": True, "enable_gradient_checkpointing": True},
            "actor": actor, "rollout": rollout, "ref": ref,
        },
        "trainer": {
            "critic_warmup": 0, "logger": ["console", "tensorboard"],
            "project_name": "riskpo_reference_" + dataset, "experiment_name": run_name,
            "n_gpus_per_node": 8 if hard else 4, "nnodes": 1,
            "save_freq": 10, "test_freq": 5,
            "total_training_steps": 500 if hard else 200,
            "total_epochs": 30 if hard else 15, "val_before_train": True,
            "default_local_dir": str(checkpoint_dir),
        },
    }

def build_lora_config(dataset, method, model_id, data_root, checkpoint_dir, run_name,
                      lora_rank=8, lora_alpha=16, train_batch_size=20, ppo_mini_batch_size=20,
                      run_profile="full"):
    """Keep original data splits; adapt training to four-GPU BF16 LoRA."""
    if run_profile not in {"smoke", "full"}:
        raise ValueError("run_profile must be smoke or full")
    if lora_rank not in {8, 16, 32, 64} or lora_alpha <= 0:
        raise ValueError("Use a supported LoRA rank (8/16/32/64) and positive alpha")
    if (train_batch_size <= 0 or ppo_mini_batch_size <= 0
            or train_batch_size % ppo_mini_batch_size != 0
            or train_batch_size % 5 != 0):
        raise ValueError("Batch must contain full bundles of 5 and be divisible by PPO mini-batch")
    cfg = build_reference_config(dataset, method, model_id, data_root, checkpoint_dir, run_name)
    cfg["data"].update(train_batch_size=train_batch_size, val_batch_size=4,
                       dataloader_num_workers=2)
    model = cfg["actor_rollout_ref"]["model"]
    model.update(lora_rank=lora_rank, lora_alpha=lora_alpha,
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"])
    actor = cfg["actor_rollout_ref"]["actor"]
    actor.update(ppo_mini_batch_size=ppo_mini_batch_size,
                 ppo_micro_batch_size_per_gpu=1, use_dynamic_bsz=False,
                 use_torch_compile=False, ulysses_sequence_parallel_size=1)
    actor["fsdp_config"].update(model_dtype="bf16", param_offload=True, optimizer_offload=True, fsdp_size=4)
    rollout = cfg["actor_rollout_ref"]["rollout"]
    rollout.update(
        tensor_model_parallel_size=1, dtype="bfloat16",
        log_prob_micro_batch_size_per_gpu=1, log_prob_use_dynamic_bsz=False,
        load_format="safetensors", layered_summon=True,
        gpu_memory_utilization=0.6, enforce_eager=True, free_cache_engine=True,
        max_num_seqs=24,
        max_num_batched_tokens=max(1024, cfg["data"]["max_prompt_length"] + cfg["data"]["max_response_length"]),
        engine_kwargs={"vllm": {"max_num_seqs": 24, "swap_space": 2}},
    )
    cfg["actor_rollout_ref"]["ref"].update(
        log_prob_micro_batch_size_per_gpu=1, log_prob_use_dynamic_bsz=False,
        ulysses_sequence_parallel_size=1)
    cfg["trainer"].update(n_gpus_per_node=4, nnodes=1, max_actor_ckpt_to_keep=1,
                          resume_mode="disable", val_before_train=False, test_freq=-1)
    if run_profile == "smoke":
        cfg["trainer"].update(total_training_steps=5, save_freq=5)
    n = cfg["actor_rollout_ref"]["rollout"]["n"]
    if (train_batch_size * n) % 4 or (ppo_mini_batch_size * n) % 4:
        raise ValueError("Global response batch and PPO minibatch must be divisible by four ranks")
    return cfg

In [ ]:
SOURCE_NOTEBOOK_NAME = 'RiskPO_Original_Qwen3_4B_4xA100.ipynb'
SOURCE_NOTEBOOK_SHA256 = 'a5ea3a2e2162b730a004dfbfa6014bb23cf87522cb190af7cf47284f77684818'
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
RUN_NAME = f"local_full_{METHOD}_{MODEL_ID.rsplit('/', 1)[-1]}_{DATASET}_{stamp}"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
CKPT_DIR = CHECKPOINT_ROOT / RUN_NAME
CONFIG = build_lora_config(
    DATASET, METHOD, MODEL_ID, DATA_DIR, CKPT_DIR, RUN_NAME,
    lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA,
    train_batch_size=TRAIN_BATCH_SIZE, ppo_mini_batch_size=PPO_MINI_BATCH_SIZE,
    run_profile="full",
)
THINKING_DATASET_SOURCE = "\nfrom copy import deepcopy\nfrom verl.utils.dataset.rl_dataset import RLHFDataset\n\ndef configured_tokenizer(tokenizer, enabled):\n    if type(enabled) is not bool:\n        raise ValueError('enable_thinking must be bool')\n    template = tokenizer.get_chat_template()\n    if 'enable_thinking' not in template:\n        raise ValueError('Tokenizer template does not support enable_thinking')\n    probe = [{'role': 'user', 'content': 'What is 1 + 1?'}]\n    expected = tokenizer.apply_chat_template(\n        probe, tokenize=False, add_generation_prompt=True, enable_thinking=enabled)\n    configured = deepcopy(tokenizer)\n    configured.chat_template = ('{% set enable_thinking = ' +\n                                ('true' if enabled else 'false') + ' %}' + template)\n    actual = configured.apply_chat_template(probe, tokenize=False, add_generation_prompt=True)\n    assert actual == expected, 'Bound template differs from explicit enable_thinking'\n    if not enabled:\n        assert actual.endswith('<think>\\n\\n</think>\\n\\n'), 'Missing Qwen3 empty thinking prefix'\n    print('Qwen3 enable_thinking:', enabled, '| prompt suffix:', repr(actual[-100:]))\n    return configured\n\nclass ThinkingModeDataset(RLHFDataset):\n    def __init__(self, data_files, tokenizer, config, processor=None):\n        if processor is not None:\n            raise ValueError('This notebook thinking override supports text-only Qwen3')\n        tokenizer = configured_tokenizer(tokenizer, config.get('enable_thinking'))\n        super().__init__(data_files=data_files, tokenizer=tokenizer, config=config, processor=None)\n"
if ENABLE_THINKING is not None:
    THINKING_DATASET_PATH = RUN_DIR / "riskpo_thinking_dataset_v1.py"
    THINKING_DATASET_PATH.write_text(THINKING_DATASET_SOURCE, encoding="utf-8")
    CONFIG["data"]["enable_thinking"] = ENABLE_THINKING
    CONFIG["data"]["custom_cls"] = {"path": str(THINKING_DATASET_PATH), "name": "ThinkingModeDataset"}
CONFIG["custom_reward_function"] = {"path": str(GSM8K_REWARD_PATH), "name": "compute_dataset_score"}
CONFIG["actor_rollout_ref"]["model"]["path"] = BASE_SNAPSHOT["path"]
TOTAL_STEPS = CONFIG["trainer"]["total_training_steps"]
CONFIG["trainer"].update(
    save_freq=TOTAL_STEPS, test_freq=TOTAL_STEPS, val_before_train=False,
    resume_mode="disable", resume_from_path=None, max_actor_ckpt_to_keep=1,
    validation_data_dir=str(RUN_DIR / "evaluation"),
)
CONFIG["actor_rollout_ref"]["actor"]["checkpoint"] = {
    "save_contents": ["model", "optimizer", "extra"],
    "load_contents": ["model", "optimizer", "extra"],
}
CONFIG_NAME = "riskpo_local_full"
CONFIG_PATH = REPO / "verl/trainer/config" / (CONFIG_NAME + ".yaml")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
shutil.copy2(CONFIG_PATH, RUN_DIR / CONFIG_PATH.name)
shutil.copy2(GSM8K_REWARD_PATH, RUN_DIR / GSM8K_REWARD_PATH.name)
ENV["TENSORBOARD_DIR"] = str(RUN_DIR / "tensorboard")
metadata = {
    "mode": "local_full", "model": MODEL_ID, "method": METHOD, "dataset": DATASET,
    "base_snapshot": BASE_SNAPSHOT, "enable_thinking": ENABLE_THINKING,
    "world_size": REQUIRED_GPUS, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
    "total_steps": TOTAL_STEPS, "train_batch_size": TRAIN_BATCH_SIZE,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    "source_notebook": SOURCE_NOTEBOOK_NAME, "source_sha256": SOURCE_NOTEBOOK_SHA256,
    "validation": "original test split once at final training step; no Telemath tuning",
    "gsm8k_scorer_version": "final_numeric_v2",
    "local_changes": ["no session limit or Drive backup", "final checkpoint", "final test evaluation", "merged model export"],
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
freeze = subprocess.check_output([str(UV), "pip", "freeze", "--python", str(PY)], text=True)
(RUN_DIR / "requirements-freeze.txt").write_text(freeze, encoding="utf-8")
print(json.dumps(CONFIG, indent=2))
print("RUN_DIR:", RUN_DIR)

## 6. Audit dataset, prompt thực tế và resolved config

In [ ]:
audit = {"config_dir": str(REPO / "verl/trainer/config"),
         "config_name": CONFIG_NAME, "run_dir": str(RUN_DIR), "dataset": DATASET}
run_py("AUDIT = " + repr(audit) + "\n" + r'''
from pathlib import Path
import hashlib
import json
import importlib.util
import pandas as pd
from hydra import initialize_config_dir, compose
from transformers import AutoTokenizer
from verl.trainer.main_ppo import create_rl_dataset

with initialize_config_dir(config_dir=AUDIT["config_dir"], version_base=None):
    cfg = compose(config_name=AUDIT["config_name"])
spec = importlib.util.spec_from_file_location("riskpo_audit_reward", cfg.custom_reward_function.path)
reward_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reward_module)
score_fn = getattr(reward_module, cfg.custom_reward_function.name)
assert reward_module.GSM8K_SCORER_VERSION == "final_numeric_v2"
tokenizer = AutoTokenizer.from_pretrained(cfg.actor_rollout_ref.model.path)
manifest = {"files": [], "usable_rows": {}}
for role, paths in [("train", cfg.data.train_files), ("evaluation", cfg.data.val_files)]:
    for filename in paths:
        path = Path(filename)
        frame = pd.read_parquet(path)
        if AUDIT["dataset"] == "gsm8k":
            expected_split = 'train' if role == 'train' else 'test'
            assert len(paths) == 1 and path.name == expected_split + '.parquet'
            assert set(frame['data_source']) == {'openai/gsm8k'}, 'Unexpected dataset in GSM8K-only run'
            assert all(info['split'] == expected_split for info in frame['extra_info']), 'Wrong GSM8K split'
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        item = {"role": role, "path": str(path), "rows": len(frame),
                "sha256": digest.hexdigest(),
                "sources": {str(k): int(v) for k, v in frame["data_source"].value_counts().items()}}
        manifest["files"].append(item)
        print(item)
        gsm = frame[frame["data_source"] == "openai/gsm8k"]
        if len(gsm):
            truth = str(gsm.iloc[0]["reward_model"]["ground_truth"])
            for response in [r"\boxed{" + truth + "}", "#### " + truth]:
                assert score_fn("openai/gsm8k", response, truth) == 1, (response, truth)
            print("GSM8K boxed/#### checks passed:", reward_module.GSM8K_SCORER_VERSION)
    dataset = create_rl_dataset(list(paths), cfg.data, tokenizer, None, is_train=(role == "train"))
    manifest["usable_rows"][role] = len(dataset)
    assert len(dataset) > 0, f"{role} empty after filtering"
    if cfg.data.get("enable_thinking") is not None:
        enabled = cfg.data.enable_thinking
        row = dataset[0]
        messages = dataset.dataframe[0][cfg.data.prompt_key]
        expected_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=enabled)
        expected_ids = tokenizer.encode(expected_text, add_special_tokens=False)
        assert list(row["raw_prompt_ids"]) == expected_ids, 'Rollout prompt does not match thinking mode'
        manifest.setdefault("thinking_checks", {})[role] = {
            "enable_thinking": enabled, "raw_prompt_ids_match": True}
        print(role, "raw_prompt_ids check passed; enable_thinking =", enabled)
assert manifest["usable_rows"]["train"] >= cfg.data.train_batch_size
(Path(AUDIT["run_dir"]) / "data_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("After prompt filtering:", manifest["usable_rows"])
''')

In [ ]:
COMMAND = [str(PY), "-m", "verl.trainer.main_ppo", "--config-name", CONFIG_NAME]
resolved = subprocess.check_output(COMMAND + ["--cfg", "job", "--resolve"], cwd=REPO, env=ENV, text=True)
(RUN_DIR / "resolved_config.yaml").write_text(resolved, encoding="utf-8")
print(resolved)
run_py("AUDIT = " + repr(audit) + "\nWORLD = " + repr(REQUIRED_GPUS) + "\nMETHOD = " + repr(METHOD) + "\n" + r'''
from hydra import initialize_config_dir, compose
import torch
with initialize_config_dir(config_dir=AUDIT["config_dir"], version_base=None):
    cfg = compose(config_name=AUDIT["config_name"])
assert torch.cuda.device_count() == cfg.trainer.n_gpus_per_node == WORLD
assert cfg.trainer.nnodes == 1
assert cfg.trainer.total_training_steps in (200, 500)
assert cfg.trainer.save_freq == cfg.trainer.test_freq == cfg.trainer.total_training_steps
assert not cfg.trainer.val_before_train
assert cfg.trainer.resume_mode == "disable"
assert not cfg.trainer.get("durable_checkpoint")
assert cfg.actor_rollout_ref.model.lora_rank > 0
assert cfg.actor_rollout_ref.actor.fsdp_config.model_dtype == "bf16"
assert cfg.actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu == 1
assert cfg.actor_rollout_ref.rollout.tensor_model_parallel_size == 1
assert cfg.data.train_batch_size * cfg.actor_rollout_ref.rollout.n % WORLD == 0
assert not cfg.actor_rollout_ref.rollout.val_kwargs.do_sample
assert cfg.actor_rollout_ref.rollout.val_kwargs.n == 1
if METHOD == "riskpo":
    assert cfg.algorithm.adv_estimator == "grpo_bundle_RVaR_quantile_tracking"
    assert cfg.algorithm.quantile_tracking
    assert cfg.actor_rollout_ref.actor.policy_loss.loss_mode == "vanilla"
else:
    assert cfg.algorithm.adv_estimator == "risk_quatro"
    assert not cfg.algorithm.quantile_tracking
    assert cfg.actor_rollout_ref.actor.policy_loss.loss_mode == "risk_quatro"
    assert cfg.actor_rollout_ref.actor.policy_loss.quatro_ratio_mode == "geometric"
print("Local full-run config checks passed; training not started yet.")
''')

## 7. Chạy đủ số bước và lưu kết quả local

Chạy cell này để bắt đầu. Không có pause sau 40 bước. Cuối run sẽ đánh giá test
rồi lưu checkpoint. Evaluation cũng cần thời gian; chưa kết thúc khi mới thấy
training step cuối đang chạy. Nếu train lỗi, giữ log và không xuất model như một
run đã hoàn thành. Không tự thay các tham số khi gặp OOM.

In [ ]:
(RUN_DIR / "command.json").write_text(json.dumps(COMMAND, indent=2), encoding="utf-8")
return_code = None
with (RUN_DIR / "train.log").open("w", encoding="utf-8", buffering=1) as log:
    process = subprocess.Popen(COMMAND, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
    except KeyboardInterrupt:
        import signal
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=30)
        raise
if return_code != 0:
    raise RuntimeError(f"Training failed: exit={return_code}; inspect {RUN_DIR / 'train.log'}")
FINAL_CHECKPOINT = CKPT_DIR / f"global_step_{TOTAL_STEPS}"
ADAPTER_DIR = FINAL_CHECKPOINT / "actor/lora_adapter"
required = ["data.pt", "actor/fsdp_config.json", "actor/lora_adapter/adapter_config.json",
            "actor/lora_adapter/adapter_model.safetensors"]
required += [f"actor/{kind}_world_size_{REQUIRED_GPUS}_rank_{rank}.pt"
             for rank in range(REQUIRED_GPUS) for kind in ("model", "optim", "extra_state")]
if not all((FINAL_CHECKPOINT / p).is_file() and (FINAL_CHECKPOINT / p).stat().st_size for p in required):
    raise RuntimeError("Full run did not produce the complete final checkpoint; do not export as completed")
fsdp = json.loads((FINAL_CHECKPOINT / "actor/fsdp_config.json").read_text())
assert fsdp["world_size"] == REQUIRED_GPUS
EVAL_FILE = RUN_DIR / "evaluation" / f"{TOTAL_STEPS}.jsonl"
if not EVAL_FILE.is_file():
    raise RuntimeError("Final evaluation output is missing")
evaluation = [json.loads(line) for line in EVAL_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
manifest = json.loads((RUN_DIR / "data_manifest.json").read_text())
assert len(evaluation) == manifest["usable_rows"]["evaluation"], "Evaluation is incomplete"
assert all(row["step"] == TOTAL_STEPS for row in evaluation)
scores = [float(row["score"]) for row in evaluation]
import math
assert scores and all(math.isfinite(s) for s in scores)
results = {
    "training_complete": True, "step": TOTAL_STEPS, "model": MODEL_ID, "method": METHOD,
    "dataset": DATASET, "evaluation_samples": len(scores), "mean_score": sum(scores) / len(scores),
    "checkpoint": str(FINAL_CHECKPOINT), "adapter": str(ADAPTER_DIR),
    "scorer": "final_numeric_v2 for GSM8K; repo default for other datasets",
    "evaluation_file": str(EVAL_FILE), "merged_model_export_complete": False,
}
if DATASET == "gsm8k":
    assert all(s in (0.0, 1.0) for s in scores)
    results.update(correct=sum(s == 1 for s in scores), accuracy=sum(scores) / len(scores))
(RUN_DIR / "results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))
print("FULL TRAINING AND FINAL EVALUATION COMPLETE. Next cell exports standalone model.")

## 8. Xuất model đầy đủ ra local

Training vẫn là LoRA. Cell này nạp đúng snapshot base đã dùng, gộp adapter bằng
PEFT trên CPU, rồi lưu model Hugging Face độc lập: không cần adapter/base riêng
khi load lại `final_model/`. Kiểm tra tensor adapter hữu hạn, safe merge,
weight shards/index và tokenizer; ghi checksum trước khi công bố export hoàn tất.

Giữ tokenizer cùng thinking mode của training. File kết quả ghi rõ bước train,
base revision và đường dẫn model. Nếu export bị ngắt, `.partial` được giữ lại để
kiểm tra, không báo thành công hoặc âm thầm ghi đè. Export không tạo điểm benchmark mới.

In [ ]:
EXPORT_MODEL_SOURCE = '"""Export a trained LoRA adapter as a standalone Hugging Face model on CPU.\n\nOnly invoked after training exits. No API service or GPU is required for export.\n"""\nimport argparse\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef export_model(base, adapter, output, thinking, step):\n    import torch\n    from peft import PeftModel\n    from safetensors import safe_open\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    base, adapter, output = Path(base).resolve(), Path(adapter).resolve(), Path(output).resolve()\n    if not (base / \'config.json\').is_file():\n        raise ValueError(\'Use the exact local base snapshot used during training\')\n    for name in (\'adapter_config.json\', \'adapter_model.safetensors\'):\n        if not (adapter / name).is_file():\n            raise FileNotFoundError(adapter / name)\n    if output.exists():\n        raise FileExistsError(f\'Refusing to overwrite exported model: {output}\')\n    partial = output.with_name(output.name + \'.partial\')\n    if partial.exists():\n        raise FileExistsError(f\'Previous partial export exists; inspect it first: {partial}\')\n    config = json.loads((adapter / \'adapter_config.json\').read_text())\n    if config.get(\'peft_type\') != \'LORA\':\n        raise ValueError(\'Expected LoRA adapter\')\n    with safe_open(str(adapter / \'adapter_model.safetensors\'), framework=\'pt\', device=\'cpu\') as tensors:\n        keys = list(tensors.keys())\n        if not keys or not any(\'lora_\' in k for k in keys):\n            raise ValueError(\'Adapter contains no LoRA tensors\')\n        for key in keys:\n            if not torch.isfinite(tensors.get_tensor(key)).all():\n                raise ValueError(f\'Non-finite adapter tensor: {key}\')\n    print(\'Loading exact training base on CPU for final merge:\', base, flush=True)\n    model = AutoModelForCausalLM.from_pretrained(\n        str(base), torch_dtype=torch.bfloat16, device_map={\'\': \'cpu\'},\n        low_cpu_mem_usage=True, attn_implementation=\'eager\', local_files_only=True,\n    )\n    peft_model = PeftModel.from_pretrained(model, str(adapter), is_trainable=False)\n    merged = peft_model.merge_and_unload(safe_merge=True)\n    if any(\'lora_\' in key for key in merged.state_dict()):\n        raise ValueError(\'Export still contains unmerged LoRA parameters\')\n    merged.save_pretrained(str(partial), safe_serialization=True, max_shard_size=\'4GB\')\n    tokenizer = AutoTokenizer.from_pretrained(str(base), local_files_only=True)\n    if thinking is not None:\n        template = tokenizer.get_chat_template()\n        expected = tokenizer.apply_chat_template(\n            [{\'role\': \'user\', \'content\': \'What is 1 + 1?\'}], tokenize=False,\n            add_generation_prompt=True, enable_thinking=thinking,\n        )\n        tokenizer.chat_template = \'{% set enable_thinking = \' + (\'true\' if thinking else \'false\') + \' %}\' + template\n        actual = tokenizer.apply_chat_template(\n            [{\'role\': \'user\', \'content\': \'What is 1 + 1?\'}], tokenize=False, add_generation_prompt=True)\n        if actual != expected:\n            raise ValueError(\'Export tokenizer does not preserve training thinking mode\')\n    tokenizer.save_pretrained(str(partial))\n    if getattr(merged, \'generation_config\', None) is not None:\n        merged.generation_config.save_pretrained(str(partial))\n    del merged, peft_model, model\n    gc.collect()\n    shards = sorted(partial.glob(\'*.safetensors\'))\n    if not shards:\n        raise ValueError(\'No full model weight shards were saved\')\n    saved_keys = set()\n    for shard in shards:\n        with safe_open(str(shard), framework=\'pt\', device=\'cpu\') as tensors:\n            keys = list(tensors.keys())\n            if any(\'lora_\' in key for key in keys):\n                raise ValueError(\'Adapter-only tensor found in merged model\')\n            if saved_keys.intersection(keys):\n                raise ValueError(\'Duplicate keys across model shards\')\n            saved_keys.update(keys)\n    index = partial / \'model.safetensors.index.json\'\n    if index.exists():\n        weight_map = json.loads(index.read_text())[\'weight_map\']\n        if set(weight_map) != saved_keys or set(weight_map.values()) != {p.name for p in shards}:\n            raise ValueError(\'Weight index and actual shards do not match\')\n    check_tokenizer = AutoTokenizer.from_pretrained(str(partial), local_files_only=True)\n    if not check_tokenizer.encode(\'export validation\'):\n        raise ValueError(\'Cannot use exported tokenizer\')\n    manifest = {\n        \'status\': \'merged_model_export_verified\', \'step\': step, \'base_snapshot\': str(base),\n        \'adapter_path\': str(adapter), \'adapter_sha256\': sha256(adapter / \'adapter_model.safetensors\'),\n        \'enable_thinking\': thinking, \'dtype\': \'bfloat16\',\n        \'validation\': \'safe LoRA merge; readable weight shards/index/tokenizer; no post-export GPU inference test\',\n        \'files\': {p.name: {\'bytes\': p.stat().st_size, \'sha256\': sha256(p)}\n                  for p in sorted(partial.iterdir()) if p.is_file()},\n    }\n    (partial / \'export_manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    partial.rename(output)\n    print(\'FULL_MODEL_EXPORTED:\', output, flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--base\', required=True)\n    parser.add_argument(\'--adapter\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--thinking\', choices=[\'true\', \'false\', \'default\'], default=\'false\')\n    parser.add_argument(\'--step\', required=True, type=int)\n    args = parser.parse_args()\n    thinking = {\'true\': True, \'false\': False, \'default\': None}[args.thinking]\n    export_model(args.base, args.adapter, args.output, thinking, args.step)\n\n\nif __name__ == \'__main__\':\n    main()\n'
export_script = RUN_DIR / "export_local_lora_model.py"
export_script.write_text(EXPORT_MODEL_SOURCE, encoding="utf-8")
FINAL_MODEL_DIR = RUN_DIR / "final_model"
thinking_arg = "default" if ENABLE_THINKING is None else str(ENABLE_THINKING).lower()
run([PY, export_script, "--base", BASE_SNAPSHOT["path"], "--adapter", ADAPTER_DIR,
     "--output", FINAL_MODEL_DIR, "--thinking", thinking_arg, "--step", TOTAL_STEPS], cwd=REPO)
export_manifest = json.loads((FINAL_MODEL_DIR / "export_manifest.json").read_text())
assert export_manifest["step"] == TOTAL_STEPS
results["merged_model_export_complete"] = True
results["final_model"] = str(FINAL_MODEL_DIR)
(RUN_DIR / "results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print("RESULTS:", RUN_DIR / "results.json")
print("FULL MODEL:", FINAL_MODEL_DIR)
print("CHECKPOINT:", FINAL_CHECKPOINT)

## Load model đã lưu và giới hạn kiểm chứng

Model độc lập nằm tại `RUN_DIR/final_model`. Dùng cùng environment đã cài:

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("/path/to/final_model", local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    "/path/to/final_model", torch_dtype="auto", device_map="auto", local_files_only=True)
```

`results.json` là kết quả final evaluation **trước merge** bằng policy cuối run.
Merge BF16 có thể có sai khác làm tròn nhỏ; chưa đánh giá lại model đã merge.
`evaluation/<step>.jsonl` giữ từng prompt/output/score, TensorBoard giữ metrics
training/validation, `train.log` giữ toàn bộ log. Telemath chưa được đánh giá ở đây.

Notebook chỉ được kiểm tra cấu trúc/config local, chưa chạy training hoặc export
model thật trong workspace Windows. Máy Linux đích phải xác nhận GPU/RAM/disk.